# GeLB on the GPU: 3-D Rayleigh-Bénard convection

This notebook compiles a **coupled D3Q19-fluid + D3Q7-temperature MRT** Lattice-Boltzmann simulation to **CUDA C++** via GeLB's GPU backend and runs it on Colab's T4 GPU. The simulation reproduces the classic Rayleigh-Bénard instability: a fluid layer heated from below and cooled from above spontaneously breaks into convection rolls once the temperature difference exceeds the critical Rayleigh number (Ra_c ≈ 1707.76 for the full-slip case; this setup uses no-slip walls, critical Ra is slightly higher).

**Before running**: switch the runtime to GPU via *Runtime → Change runtime type → T4 GPU*. The first cell will refuse to run on a CPU-only runtime.

**This is an advanced example.** If you're new to GeLB, start with the [`gelb_intro_2d.ipynb`](./gelb_intro_2d.ipynb) notebook (2-D Taylor-Green vortex, 70 lines of source). The example below showcases what the language can do at the complexity end of the spectrum — multiple coupled lattices, MRT collision with d'Humières moment ordering, non-trivial wall boundary conditions, and Strang-split buoyancy coupling — and is ~555 lines of GeLB source, the repository's [`rayleigh_benard_3d_mrt.gelb`](https://github.com/dchirila/gelb/blob/v1.0.0/examples/lbm/3d/rayleigh_benard_3d_mrt.gelb).

You will:

1. Verify the GPU is available.
2. Install NetCDF-C (the CUDA toolkit is preinstalled on Colab GPU runtimes).
3. Install GeLB.
4. Download the RB-3D `.gelb` source from the release.
5. Compile it with `gelbc --backend gpu_cuda_cpp`.
6. Run the simulation on the GPU.
7. Visualise the temperature field and the emergence of convection rolls.

Expected runtime: a few minutes on the first run, most of it installing NetCDF and GeLB.

## 1. Confirm a GPU is attached

In [ ]:
import subprocess
try:
    out = subprocess.check_output(['nvidia-smi', '-L'], text=True)
    print(out)
except (FileNotFoundError, subprocess.CalledProcessError):
    raise RuntimeError(
        'No GPU detected. Switch the runtime via '
        'Runtime → Change runtime type → T4 GPU, then re-run.')

## 2. Confirm `nvcc` is on PATH

Colab GPU runtimes ship with the CUDA toolkit preinstalled, so `nvcc` should already be on `PATH`. The cell below confirms it.

In [ ]:
!nvcc --version | head -4

## 3. Install NetCDF-C

For the gauge output. The CUDA C++ backend links against the same NetCDF-C library that the CPU C++ backends use — `libnetcdf-dev` from apt is sufficient.

In [ ]:
!apt-get install -y -qq libnetcdf-dev > /dev/null
!dpkg -l | grep -E '^ii\s+libnetcdf'

## 4. Install GeLB

GeLB 1.0.0, the release described in the SoftwareX paper, installs from its tag on GitHub (about 30 s), as in the [2-D CPU notebook](./gelb_intro_2d.ipynb):

In [ ]:
!pip install -q "git+https://github.com/dchirila/gelb.git@v1.0.0"

# gelb-run starts the compiler and its progress monitor through a Python
# interpreter that can import gelb: name this notebook's own.
import os, sys
os.environ["GELB_PYTHON"] = sys.executable

In [ ]:
!which gelbc gelb-run
!python -c "import importlib.metadata as m; print('gelb', m.version('gelb'))"

GeLB source is shown with syntax highlighting by the Pygments lexer that the GeLB manuals use:

In [ ]:
!curl -sSfL -o gelb_lexer.py https://raw.githubusercontent.com/dchirila/gelb/v1.0.0/docs/_ext/gelb_lexer.py

from gelb_lexer import GelbLexer
from pygments import highlight
from pygments.formatters import HtmlFormatter
from IPython.display import HTML, display

def show_gelb(path, max_height=None):
    """Display a .gelb file with syntax highlighting and line numbers."""
    with open(path) as f:
        html = highlight(f.read(), GelbLexer(),
                         HtmlFormatter(noclasses=True, style="friendly", linenos="inline",
                                       prestyles="font-family: monospace; font-size: 13px; line-height: 125%;"))
    box = f"max-height: {max_height}px; overflow: auto;" if max_height else ""
    display(HTML(f'<div style="{box}">{html}</div>'))

## 5. Get the Rayleigh-Bénard GeLB program

The program is [`examples/lbm/3d/rayleigh_benard_3d_mrt.gelb`](https://github.com/dchirila/gelb/blob/v1.0.0/examples/lbm/3d/rayleigh_benard_3d_mrt.gelb) from the release, a GeLB port of the reference Fortran solver used in the author's 2017 PhD-thesis research runs. Key ingredients:

- **Two coupled lattices**: `f` (D3Q19) for the fluid momentum, `g` (D3Q7) for the temperature scalar.
- **MRT collision** for both, with the d'Humières moment ordering for the fluid.
- **Strang-split buoyancy**: half the Boussinesq forcing is added to the jy moment before the collision, half after.
- **Half-way fluid bounce-back** at the top/bottom walls (no-slip), with the Yoshida anti-bounce-back for the thermal BC.
- **Periodic** in x and z; walls in y.
- **IC**: quadratic hydrostatic δρ + linear T profile + O(10⁻⁸) random noise + a single delta perturbation at `(NX/2, 2, *)` to seed the critical eigenmode quickly.

The cell below downloads it and shows it with syntax highlighting in a scrollable box; to change it, open `rb_3d.gelb` in Colab's Files pane (or pass `-D NAME=VALUE` to `gelb-run`, see *Next steps*).

In [ ]:
!curl -sSfL -o rb_3d.gelb https://raw.githubusercontent.com/dchirila/gelb/v1.0.0/examples/lbm/3d/rayleigh_benard_3d_mrt.gelb
show_gelb('rb_3d.gelb', max_height=500)

## 6. Compile, build, and run on the GPU

`gelb-run` invokes `gelbc --backend gpu_cuda_cpp`, builds with `nvcc -O3 -arch=sm_75 -std=c++17 ...` (T4 = `sm_75`), and launches the executable on the T4.

With the default grid (`NY=16`, `NX=NZ=32`, `RA=10000`) the seed perturbation first decays; convection sets in around step 10 000 and the rolls are fully developed by step 14 000. The run below takes 16 000 steps, a few seconds of GPU time; the first cell run spends most of its time in the nvcc build.

In [ ]:
import os
os.environ['GELB_NVCC_ARCH'] = '-arch=sm_75'  # T4 = compute capability 7.5
!gelb-run -b gpu_cuda_cpp -n 16000 --write-every 400 -v rb_3d.gelb

## 7. Visualise the convection rolls

NetCDF output has shape `(time, z, y, x)` (Python reverses the Fortran dimension order). The two wall rows, `y = 0` and `y = NY-1`, are not fluid and hold no values (NaN). A vertical cross-section at `z = NZ/2` shows the temperature field evolving from the uniform-linear initial state into characteristic rolls.

In [ ]:
import netCDF4, numpy as np, matplotlib.pyplot as plt

ds = netCDF4.Dataset('rb_3d.nc')
print('Variables:', list(ds.variables.keys()))
print('Shape of T:', ds.variables['T'].shape, '  # (time, z, y, x)')

T = ds.variables['T'][:]
nt, nz, ny, nx = T.shape
z_mid = nz // 2
write_every = 400

# 4-panel time series: initial, 1/3, 2/3, final.
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
indices = [0, nt // 3, 2 * nt // 3, nt - 1]
vmin, vmax = 0.0, 1.0
for ax, idx in zip(axes, indices):
    im = ax.imshow(T[idx, z_mid], origin='lower',
                    cmap='RdYlBu_r', vmin=vmin, vmax=vmax,
                    aspect='auto')
    ax.set_title(f'T(x, y) at z = NZ/2, step {idx * write_every}')
    ax.set_xlabel('x'); ax.set_ylabel('y')
plt.colorbar(im, ax=axes, fraction=0.015, pad=0.02, label='Temperature')
plt.suptitle('Rayleigh-Bénard roll development — vertical cross-section', y=1.02)
plt.show()

### 7b. Time-averaged vertical profile

Averaging the temperature over x and z (and, optionally, over the second half of the simulation for the time-steady average) gives the mean vertical profile. In the purely-conductive solution this would be linear between T_hot=1 and T_cold=0 at the two walls, which the half-way bounce-back places half a cell inside the wall rows (y = 0.5 and y = NY-1.5); convection flattens the interior, producing a characteristic S-shape.

In [ ]:
# Mean vertical profile over the fluid rows, averaged over x, z, and the last third of the run.
y = np.arange(1, ny - 1)                          # rows 0 and NY-1 are walls
T_mean = T[2 * nt // 3 :, :, 1 : ny - 1, :].mean(axis=(0, 1, 3))
T_cond = 1.0 - (y - 0.5) / (ny - 2)               # purely-conductive reference

plt.figure(figsize=(4.5, 5))
plt.plot(T_mean, y, 'o-', label='GeLB-CUDA time-averaged')
plt.plot(T_cond, y, 'k--', label='purely-conductive (no convection)')
plt.xlabel('T (x, z, t-averaged)')
plt.ylabel('y (vertical)')
plt.title('Mean temperature profile')
plt.legend(loc='lower left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Next steps

- **Turn up Ra** — `gelb-run` (and `gelbc`) accept `-D RA=50000`, `-D RA=100000`, etc. at compile time. The flow becomes increasingly turbulent and the profile flattens further.
- **Turn up resolution** — `-D NY=32` (and the derived `NX=NZ=2*NY`) gives finer rolls but pushes the runtime up by roughly 8×.
- **Measure Nusselt number** — the ratio `Nu = 1 + 〈u_y·T〉/(DIFFUSIVITY · (T_hot − T_cold) / H)` is the standard RB heat-transport metric. Computable from the NetCDF above.
- **Check MLUPS** — `gelb-run`'s output prints mega-lattice-updates-per-second. This grid is small (16 384 cells), so the GPU is far from busy; try `-D NY=32`.
- **Read the CUDA C++ source** — `gelbc -b gpu_cuda_cpp -o rb_3d.cu rb_3d.gelb` and open `rb_3d.cu`. Every kernel is explicit and readable.

Documentation: the [GeLB User Tutorial](https://github.com/dchirila/gelb/blob/v1.0.0/docs/pdf/GeLB_User_Tutorial.pdf) (PDF); repository and issue tracker at [github.com/dchirila/gelb](https://github.com/dchirila/gelb).